# 04.10 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/10-regularisation/solutions.ipynb)

## Exercise 1

α = 20: m = 500 / (40 + 20) = **8.33**, c = 100 − 8.33 × 6 = **50**.

α = 160: m = 500 / 200 = **2.5**, c = 100 − 15 = **85**.

As α → ∞, m → 0 and c → ȳ = 100: the line becomes flat at the mean. Infinite regularisation is the "predict the mean" baseline from 02.1.

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

x = np.array([2, 4, 6, 8, 10.])            # visits last month
y = np.array([50, 80, 90, 130, 150.])      # spend next month, £
X5 = x.reshape(-1, 1)

from sklearn.linear_model import Ridge
for a in [20, 160, 1e6]:
    r = Ridge(alpha=a).fit(X5, y)
    print(f"α={a:>9}: m={r.coef_[0]:.3f}, c={r.intercept_:.2f}")

α=       20: m=8.333, c=50.00
α=      160: m=2.500, c=85.00
α=1000000.0: m=0.000, c=100.00


## Exercise 2

In [2]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

cust20 = pd.read_csv(data_path("shopwave_customers_small.csv"))
X20 = cust20[["visits_last_month", "items_last_month", "days_since_last"]].copy()
rng = np.random.default_rng(7)
for i in range(1, 9):                       # eight columns of pure noise
    X20[f"noise_{i}"] = rng.normal(size=len(X20))
y20 = cust20["spend_next_month"]
X20.round(2).head()

,visits_last_month,items_last_month,days_since_last,noise_1,noise_2,noise_3,noise_4,noise_5,noise_6,noise_7,noise_8
0,2,2,55,0.00,-1.84,0.11,0.20,-0.43,-0.30,0.09,1.10
1,11,8,44,0.30,-0.24,0.06,-0.46,-0.30,-0.90,-0.59,-0.54
2,10,6,22,-0.27,-1.27,-1.23,0.13,0.35,0.16,-0.12,-0.05
3,7,6,58,-0.89,0.27,0.08,-1.19,-0.12,2.24,-2.00,-0.79
4,7,5,25,-0.45,0.16,1.36,-0.58,-0.20,-0.83,-1.13,-0.63


In [3]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
raw = LassoCV(cv=5, random_state=0, max_iter=50_000).fit(X20, y20)
scaled = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0, max_iter=50_000)).fit(X20, y20)
print(X20.std().round(2).to_dict())
pd.DataFrame({"unscaled": raw.coef_, "scaled": scaled[-1].coef_}, index=X20.columns).round(2)

{'visits_last_month': 4.03, 'items_last_month': 2.22, 'days_since_last': 15.82, 'noise_1': 0.78, 'noise_2': 0.87, 'noise_3': 0.92, 'noise_4': 0.91, 'noise_5': 0.86, 'noise_6': 0.8, 'noise_7': 1.06, 'noise_8': 0.83}


,unscaled,scaled
visits_last_month,13.44,50.29
items_last_month,5.76,13.59
days_since_last,-0.66,-8.73
noise_1,0.00,0.00
noise_2,0.00,0.00
noise_3,0.00,0.00
noise_4,0.00,-0.00
noise_5,-0.00,-0.00
noise_6,-0.00,-0.00
noise_7,-0.00,-0.00


Standard deviations: `days_since_last` 15.8, `visits` 4.0, `items` 2.2, noise columns about 0.9. Unscaled coefficients are in "£ per unit of that column", so `days_since_last` only needs −0.66 £/day while `items` needs 5.76 £/item. Lasso's penalty is on coefficient size, so unscaled it penalises small-spread features (items) far more than large-spread ones (days) for the same usefulness.

On this table the same noise columns happen to be zeroed either way. That is luck of units: on another dataset a useful feature measured in small units is the one that gets removed. Always scale before Ridge or Lasso, inside a pipeline so it is fitted per fold.

## Exercise 3

In [4]:
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import cross_val_score, KFold
X50 = X20[["visits_last_month", "items_last_month", "days_since_last"]].copy()
rng2 = np.random.default_rng(11)
for i in range(30):
    X50[f"noise_{i}"] = rng2.normal(size=len(X50))
cv = KFold(5, shuffle=True, random_state=0)
for name, m in [("least squares", LinearRegression()), ("RidgeCV", RidgeCV(alphas=np.logspace(-2, 3, 30))),
                ("LassoCV", LassoCV(cv=5, random_state=0, max_iter=50_000))]:
    mae = -cross_val_score(make_pipeline(StandardScaler(), m), X50, y20, cv=cv, scoring="neg_mean_absolute_error").mean()
    print(f"{name:<14} CV MAE £{mae:.2f}")

least squares  CV MAE £37.15
RidgeCV        CV MAE £37.91
LassoCV        CV MAE £15.32


With 33 features and 16 training rows per fold, least squares can fit the training data perfectly with noise alone, and its held-out error more than doubles (£37). **Ridge does not rescue it** (£38): it shrinks all 33 coefficients but keeps every noise column. **Lasso** drops the noise and stays at £15. When most features are useless, L1 is the right tool.

## Exercise 4

In [5]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")                       # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])
features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

cust["avg_basket"] = cust["spend"] / cust["orders"]
cust["orders_per_month"] = cust["orders"] / (cust["tenure_days"] / 30).clip(lower=1)
cust["log_spend"] = np.log1p(cust["spend"])
country = past.groupby("Customer ID")["Country"].agg(lambda s: s.mode()[0])
cust["country"] = country.reindex(cust.index)
cust["is_uk"] = (cust["country"] == "United Kingdom").astype(int)
weekend = past.assign(wk=past["InvoiceDate"].dt.dayofweek >= 5).groupby("Customer ID")["wk"].mean()
cust["share_weekend"] = weekend.reindex(cust.index)
autumn = past[past["InvoiceDate"] >= "2011-06-01"].groupby("Customer ID")["Invoice"].nunique()
cust["orders_last_3m"] = autumn.reindex(cust.index).fillna(0)

churn_cols = ["orders", "spend", "products", "tenure_days", "days_since_last", "avg_basket",
              "orders_per_month", "orders_last_3m", "is_uk", "share_weekend"]
Xc = cust[churn_cols].copy()
skewed = ["orders", "spend", "products", "avg_basket", "orders_per_month", "orders_last_3m"]
Xc[skewed] = np.log1p(Xc[skewed])
from sklearn.model_selection import train_test_split
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, cust["churned"], test_size=0.25,
                                                        random_state=42, stratify=cust["churned"])

3,317 customers, churn rate 43.6%


In [6]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
m = make_pipeline(StandardScaler(), LogisticRegressionCV(Cs=20, penalty="l1", solver="liblinear",
                                                         scoring="roc_auc", cv=5))
m.fit(Xc_train, yc_train)
lr = m[-1]
kept = [c for c, w in zip(churn_cols, lr.coef_[0]) if w != 0]
print(f"chosen C = {lr.C_[0]:.4f}; kept {len(kept)} features: {kept}")
print(f"test AUC: {roc_auc_score(yc_test, m.predict_proba(Xc_test)[:, 1]):.3f}")

chosen C = 0.6158; kept 9 features: ['orders', 'products', 'tenure_days', 'days_since_last', 'avg_basket', 'orders_per_month', 'orders_last_3m', 'is_uk', 'share_weekend']
test AUC: 0.743


CV picks the C with the highest AUC (about 0.6), which keeps nine features; test AUC 0.743. CV does not reward simplicity on its own. If a smaller model matters to you, a common rule is to take the strongest penalty whose CV score is within a hair of the best: from the lesson's table, C = 0.01 keeps five features for a CV AUC of 0.741 instead of 0.742.